# Shared Pipeline & Baselines




In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd



Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

DATA_PATH = "/content/drive/MyDrive/GrainWise_Project_ML/Rice_Cammeo_Osmancik.arff"
import os; print("Exists:", os.path.exists(DATA_PATH))

Exists: True


## The shared module
Running this cell writes `rice_shared.py` into the Colab session.

In [10]:
%%writefile rice_shared.py

from __future__ import annotations
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RepeatedStratifiedKFold, cross_validate)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE   = 42
TEST_SIZE      = 0.20
POSITIVE_CLASS = "Cammeo"          # the premium variety = class 1
DATASET_DOI    = "10.24432/C5MW4Z"

# <<< EACH MEMBER: set this to YOUR Drive path, or do rs.DATA_PATH = "..." after import >>>
DATA_PATH = "/content/drive/MyDrive/GrainWise_Project_ML/Rice_Cammeo_Osmancik.arff"

EXPECTED       = ["Area", "Perimeter", "Major_Axis_Length", "Minor_Axis_Length",
                  "Eccentricity", "Convex_Area", "Extent", "Class"]
RAW_FEATURES   = ["Area", "Perimeter", "Major_Axis_Length", "Minor_Axis_Length",
                  "Eccentricity", "Convex_Area", "Extent"]
RATIO_FEATURES = ["Aspect_Ratio", "Roundness", "Solidity"]

# EVIDENCE (Notebook 1, training partition):
#   Area ~ Convex_Area r=0.9989; VIF 30,902 / 32,541. Only Extent has VIF < 10.
#   Aspect_Ratio ~ Eccentricity r=0.9828, IDENTICAL univariate AUC 0.8571 - eccentricity
#   is a deterministic transform of the axis ratio, so Aspect_Ratio is engineered
#   redundancy. Dropped from every set except "all", where it stays visible as a finding.
FEATURE_SETS = {
    "all":          RAW_FEATURES + RATIO_FEATURES,
    "raw":          RAW_FEATURES,
    # scale-invariant: no pixel-unit features, so it transfers to another imaging rig
    "invariant":    ["Eccentricity", "Extent", "Roundness", "Solidity"],
    # resolved at runtime by greedy VIF elimination on the TRAINING partition only
    "decorrelated": None,
}

VIF_THRESHOLD = 10.0


def greedy_vif_select(X: pd.DataFrame, threshold: float = VIF_THRESHOLD) -> list:
    """Drop the highest-VIF feature until all remaining are under threshold.
    Fitted on the training partition only, and deterministic, so every member
    gets the same list without passing files around."""
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from sklearn.preprocessing import StandardScaler
    cols = list(X.columns)
    while len(cols) > 1:
        Z = StandardScaler().fit_transform(X[cols])
        vifs = []
        for i in range(len(cols)):
            try:
                v = variance_inflation_factor(Z, i)
                vifs.append(np.inf if not np.isfinite(v) else v)
            except Exception:
                vifs.append(np.inf)
        worst = int(np.argmax(vifs))
        if vifs[worst] <= threshold:
            break
        cols.pop(worst)
    return cols

SCORING = {"accuracy": "accuracy", "balanced_accuracy": "balanced_accuracy",
           "precision": "precision", "recall": "recall",
           "f1": "f1", "roc_auc": "roc_auc"}

CV       = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=RANDOM_STATE)
INNER_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

_cache = {}


def load_raw(path: str | None = None) -> pd.DataFrame:
    """Reads .arff / .csv / .xlsx and normalises columns. Each member sets
    rs.DATA_PATH to their own Drive path after importing."""
    path = path or DATA_PATH
    if _cache.get("path") != path:
        p = str(path).lower()
        if p.endswith(".arff"):
            cols, rows, in_data = [], [], False
            with open(path, "r", errors="ignore") as f:
                for line in f:
                    s = line.strip()
                    if not s or s.startswith("%"):
                        continue
                    low = s.lower()
                    if low.startswith("@attribute"):
                        cols.append(s.split()[1].strip("'\""))
                    elif low.startswith("@data"):
                        in_data = True
                    elif in_data:
                        rows.append([v.strip().strip("'\"") for v in s.split(",")])
            d = pd.DataFrame(rows, columns=cols)
        elif p.endswith((".xlsx", ".xls")):
            d = pd.read_excel(path)
        else:
            d = pd.read_csv(path)

        d.columns = [str(c).strip().replace(" ", "_") for c in d.columns]
        d = d.rename(columns={c: e for c in d.columns for e in EXPECTED
                              if c.lower() == e.lower()})
        missing = [c for c in EXPECTED if c not in d.columns]
        if missing:
            raise ValueError(f"Missing columns {missing}.\nFound: {list(d.columns)}")
        d = d[EXPECTED].copy()
        for c in EXPECTED[:-1]:
            d[c] = pd.to_numeric(d[c], errors="coerce")
        d["Class"] = d["Class"].astype(str).str.strip().str.capitalize()
        _cache["path"], _cache["raw"] = path, d
    return _cache["raw"].copy()


def add_ratio_features(d: pd.DataFrame) -> pd.DataFrame:
    """Row-wise arithmetic only — safe to compute before the split."""
    d = d.copy()
    d["Aspect_Ratio"] = d["Major_Axis_Length"] / d["Minor_Axis_Length"]
    d["Roundness"]    = 4 * np.pi * d["Area"] / (d["Perimeter"] ** 2)
    d["Solidity"]     = d["Area"] / d["Convex_Area"]
    return d


def get_split(feature_set: str = "all", seed: int = RANDOM_STATE):
    """Identical partition for every member. Returns X_train, X_test, y_train, y_test."""
    if feature_set not in FEATURE_SETS:
        raise KeyError(f"feature_set must be one of {list(FEATURE_SETS)}")
    d = add_ratio_features(load_raw())
    y = (d["Class"] == POSITIVE_CLASS).astype(int)
    cols = FEATURE_SETS[feature_set]
    if cols is None:                      # "decorrelated" - derive from TRAIN only
        key = ("decorr", seed)
        if key not in _cache:
            Xtr_full, _, _, _ = train_test_split(
                d[FEATURE_SETS["all"]], y, test_size=TEST_SIZE,
                stratify=y, random_state=seed)
            _cache[key] = greedy_vif_select(Xtr_full)
        cols = _cache[key]
    return train_test_split(d[cols], y, test_size=TEST_SIZE,
                            stratify=y, random_state=seed)


def make_pipeline(model, scale: bool = True) -> Pipeline:
    """scale=True for SVM / kNN / LogReg; scale=False for trees and ensembles."""
    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)


def score(model, X, y, name=None, scale=True, cv=CV):
    """Returns (summary_row_dict, per_fold_arrays). Same schema for all four members."""
    pipe = make_pipeline(clone(model), scale=scale)
    res = cross_validate(pipe, X, y, cv=cv, scoring=SCORING,
                         n_jobs=-1, return_train_score=False)
    row = {"model": name or model.__class__.__name__, "scaled": scale}
    for m in SCORING:
        row[f"{m}_mean"] = res["test_" + m].mean()
        row[f"{m}_std"]  = res["test_" + m].std()
    row["fit_time_s"] = res["fit_time"].mean()
    return row, res


def save_results(row, res, path):
    """Write one member's result so the comparison table assembles itself."""
    pd.DataFrame([row]).to_csv(path, index=False)
    np.save(path.replace(".csv", "_folds.npy"), res["test_accuracy"])
    print("saved:", path)


Overwriting rice_shared.py


## Smoke test — does it reproduce Notebook 1's split exactly?

In [ ]:
import rice_shared as rs
rs.DATA_PATH = DATA_PATH

X_train, X_test, y_train, y_test = rs.get_split("all")
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train balance:", round(y_train.mean(), 4), " Test balance:", round(y_test.mean(), 4))
print("Features:", list(X_train.columns))
print("\nThese numbers MUST match Notebook 1 Step 4a. If they don't, stop and tell me.")

Train: (3048, 10)  Test: (762, 10)
Train balance: 0.4278  Test balance: 0.4278
Features: ['Area', 'Perimeter', 'Major_Axis_Length', 'Minor_Axis_Length', 'Eccentricity', 'Convex_Area', 'Extent', 'Aspect_Ratio', 'Roundness', 'Solidity']

These numbers MUST match Notebook 1 Step 4a. If they don't, stop and tell me.


### The decorrelated set, derived from evidence
Notebook 1 found VIF up to 32,541 and only `Extent` under 10. Rather than hand-pick
which size measures to drop, the harness runs greedy VIF elimination **on the training
partition only** and every member gets the same deterministic list. Print it so it
goes in the decision log.

In [ ]:
!pip install statsmodels -q
Xd, _, yd, _ = rs.get_split("decorrelated")
print("Greedy VIF elimination kept:", list(Xd.columns))
print("Dropped:", [c for c in rs.FEATURE_SETS["all"] if c not in Xd.columns])

Greedy VIF elimination kept: ['Minor_Axis_Length', 'Eccentricity', 'Extent', 'Solidity']
Dropped: ['Area', 'Perimeter', 'Major_Axis_Length', 'Convex_Area', 'Aspect_Ratio', 'Roundness']


### Confirming the engineered-redundancy finding


In [ ]:
chk = rs.add_ratio_features(rs.load_raw())
derived = np.sqrt(1 - 1 / chk["Aspect_Ratio"] ** 2)
print("max |derived Eccentricity - actual|:", float((derived - chk["Eccentricity"]).abs().max()))
print("Spearman (rank) correlation:",
      round(float(chk["Aspect_Ratio"].corr(chk["Eccentricity"], method="spearman")), 6))
print("\n-> Aspect_Ratio carries no information Eccentricity does not. Dropped from")
print("   every feature set except 'all', where it stays visible as the finding.")

max |derived Eccentricity - actual|: 5.520946699810736e-08
Spearman (rank) correlation: 1.0

-> Aspect_Ratio carries no information Eccentricity does not. Dropped from
   every feature set except 'all', where it stays visible as the finding.


## Baselines
Two of them, and both matter.

- **Majority class** — the floor any claim of skill must clear.
- **Single-feature threshold (depth-1 tree)** — the honest baseline.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, export_text

rows = []

row, res = rs.score(DummyClassifier(strategy="most_frequent"),
                    X_train, y_train, name="Baseline: majority class", scale=False)
rows.append(row)

stump = DecisionTreeClassifier(max_depth=1, random_state=rs.RANDOM_STATE)
row, res_stump = rs.score(stump, X_train, y_train,
                          name="Baseline: 1-feature threshold", scale=False)
rows.append(row)

base = pd.DataFrame(rows)
display(base[["model", "accuracy_mean", "accuracy_std", "f1_mean", "roc_auc_mean"]].round(4))

stump.fit(X_train, y_train)
print("\nWhat the single-threshold baseline actually uses:")
print(export_text(stump, feature_names=list(X_train.columns)))

,model,accuracy_mean,accuracy_std,f1_mean,roc_auc_mean
0,Baseline: majority class,0.5722,0.0015,0.0000,0.5000
1,Baseline: 1-feature threshold,0.9290,0.0137,0.9168,0.9272



What the single-threshold baseline actually uses:
|--- Major_Axis_Length <= 191.82
|   |--- class: 0
|--- Major_Axis_Length >  191.82
|   |--- class: 1



## Feature-set comparison
The scale-invariance trade-off, measured rather than asserted — does dropping the
pixel-unit features cost us anything?

In [ ]:
from sklearn.linear_model import LogisticRegression

fs_rows, n_feats = [], []
for fs in ["all", "raw", "invariant", "decorrelated"]:
    Xtr, _, ytr, _ = rs.get_split(fs)
    r, _ = rs.score(LogisticRegression(max_iter=2000), Xtr, ytr, name=fs, scale=True)
    fs_rows.append(r); n_feats.append(Xtr.shape[1])

fs_df = pd.DataFrame(fs_rows).rename(columns={"model": "feature_set"})
fs_df["n_features"] = n_feats
display(fs_df[["feature_set", "n_features", "accuracy_mean", "accuracy_std",
               "precision_mean", "roc_auc_mean"]].round(4))

,feature_set,n_features,accuracy_mean,accuracy_std,precision_mean,roc_auc_mean
0,all,10,0.9312,0.0113,0.9257,0.9799
1,raw,7,0.9316,0.0110,0.9218,0.9791
2,invariant,4,0.7924,0.0196,0.7546,0.8632
3,decorrelated,4,0.9296,0.0105,0.9192,0.9795


## Smoke test — all four members' model configs run

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold

quick = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
configs = [("M1 LogisticRegression", LogisticRegression(max_iter=2000), True),
           ("M2 RandomForest",       RandomForestClassifier(random_state=42, n_jobs=-1), False),
           ("M3 SVM (RBF)",          SVC(probability=True, random_state=42), True),
           ("M4 kNN",                KNeighborsClassifier(), True)]

for name, mdl, sc in configs:
    r, _ = rs.score(mdl, X_train, y_train, name=name, scale=sc, cv=quick)
    print(f"{name:26s} scaled={str(sc):5s} acc={r['accuracy_mean']:.4f}")
print("\nAll four configs run. Harness is ready to hand over.")

M1 LogisticRegression      scaled=True  acc=0.9314
M2 RandomForest            scaled=False acc=0.9236
M3 SVM (RBF)               scaled=True  acc=0.9304
M4 kNN                     scaled=True  acc=0.9213

All four configs run. Harness is ready to hand over.


In [11]:
!cp /content/rice_shared.py "/content/drive/MyDrive/GrainWise_Project_ML/"
print("saved to Drive")

saved to Drive
